# Chains in LangChain

Chain:
- is the key building block of LangChain
- Combines LLM Model with a prompt 
- Building block to carry out a sequence of operations on your text or data

Types of Chains:
- LLM Chain
- Simple Sequential Chain
- Sequential Chain
- Router Chain

 

#### _Helper Function to Connect to LLM:_

    import os
    os.environ['ACCESS_TOKEN_NAME'] = 'insert_access_token'

    from openai import OpenAI

    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=os.environ["HF_ACCESS_TOKEN"],
    )

    def get_completion(messages, model="zai-org/GLM-5.3:fireworks-ai", temperature=0):
        response = client.chat.completions.create(
            model= model,
            messages = messages,
            temperature = temperature,
        )

        return response.choices[0].message.content

#### _Load Data to use:_


- Load pandas dataframe (data structure) by reading the csv file 
- View first 5 rows of the dataframe using head method
- This example has 2 columns as product name and review

#### _LLM Chain:_


Import these 3 from LangChain -
- Language model
- Prompt template from prompts
- LLM Chain from chains

    from langchain.chat_models import ChatOpenAI
    from langchain.prompts import ChatPromptTemplate
    from langchain.chains import LLMChaina

Call LLM with higher temperature to get more creative responses:

    llm = ChatOpenAI(temperature=0.9, model=llm_model)

Create Prompt:

    prompt = ChatPromptTemplate.from_template(
        "What is the best name to describe \
        a company that makes {product}?"
    )

LLM Chain:

    chain = LLMChain(llm=llm, prompt=prompt)

    product = "Queen Size Sheet Set"
    chain.run(product)

#### _Simple Sequential Chain:_


- It is another type of chain that runs a sequence of chain one after the other
- Takes chains where each have single input and single output
- Used when there is a single input and single output

Import Simple Sequencial chain from LangChain chains:

    from langchain.chains import SimpleSequentialChain

Chain 1:

    llm = ChatOpenAI(temperature=0.9, model=llm_model)

    # prompt template 1
    first_prompt = ChatPromptTemplate.from_template(
        "What is the best name to describe \
        a company that makes {product}?"
    )

    # Chain 1
    chain_one = LLMChain(llm=llm, prompt=first_prompt)

Chain 2:

    # prompt template 2
    second_prompt = ChatPromptTemplate.from_template(
        "Write a 20 words description for the following \
        company:{company_name}"
    )
    # chain 2
    chain_two = LLMChain(llm=llm, prompt=second_prompt)

Combine the Chains in a simple sequence chain:

    overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                                verbose=True
                                                )

    overall_simple_chain.run(product)

#### _Sequential Chain:_

- Can take multiple input variables in chain so it is import to have clear variable output key names
- Used when there is multiple inputs or outputs
- Ensure input and output variable names are specific and used properly in prompts
- Specify input and output variables

Import Sequencial chain from LangChain chains:

    from langchain.chains import SequentialChain

Chain 1:

    llm = ChatOpenAI(temperature=0.9, model=llm_model)

    # prompt template 1: translate to english
    first_prompt = ChatPromptTemplate.from_template(
        "Translate the following review to english:"
        "\n\n{Review}"
    )
    # chain 1: input= Review and output= English_Review
    chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                        output_key="English_Review"
                        )


Chain 2:

    second_prompt = ChatPromptTemplate.from_template(
        "Can you summarize the following review in 1 sentence:"
        "\n\n{English_Review}"
    )
    # chain 2: input= English_Review and output= summary
    chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                        output_key="summary"
                        )


Chain 3:

    # prompt template 3: translate to english
    third_prompt = ChatPromptTemplate.from_template(
        "What language is the following review:\n\n{Review}"
    )
    # chain 3: input= Review and output= language
    chain_three = LLMChain(llm=llm, prompt=third_prompt,
                        output_key="language"
                        )


Chain 4:


    # prompt template 4: follow up message
    fourth_prompt = ChatPromptTemplate.from_template(
        "Write a follow up response to the following "
        "summary in the specified language:"
        "\n\nSummary: {summary}\n\nLanguage: {language}"
    )
    # chain 4: input= summary, language and output= followup_message
    chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                        output_key="followup_message"
                        )


Combine the Chains in a sequence chain:

    # overall_chain: input= Review 
    # and output= English_Review,summary, followup_message
    overall_chain = SequentialChain(
        chains=[chain_one, chain_two, chain_three, chain_four],
        input_variables=["Review"],
        output_variables=["English_Review", "summary","followup_message"],
        verbose=True
    )

    review = df.Review[5]
    overall_chain(review)

#### _Router Chain:_


- Route input to chain depending on the output
- Used to make a more complicated chain
- Change prompt based on input

_Example:_

    physics_template = """You are a very smart physics professor. \
    You are great at answering questions about physics in a concise\
    and easy to understand manner. \
    When you don't know the answer to a question you admit\
    that you don't know.

    Here is a question:
    {input}"""


    math_template = """You are a very good mathematician. \
    You are great at answering math questions. \
    You are so good because you are able to break down \
    hard problems into their component parts, 
    answer the component parts, and then put them together\
    to answer the broader question.

    Here is a question:
    {input}"""

    history_template = """You are a very good historian. \
    You have an excellent knowledge of and understanding of people,\
    events and contexts from a range of historical periods. \
    You have the ability to think, reflect, debate, discuss and \
    evaluate the past. You have a respect for historical evidence\
    and the ability to make use of it to support your explanations \
    and judgements.

    Here is a question:
    {input}"""


    computerscience_template = """ You are a successful computer scientist.\
    You have a passion for creativity, collaboration,\
    forward-thinking, confidence, strong problem-solving capabilities,\
    understanding of theories and algorithms, and excellent communication \
    skills. You are great at answering coding questions. \
    You are so good because you know how to solve a problem by \
    describing the solution in imperative steps \
    that a machine can easily interpret and you know how to \
    choose a solution that has a good balance between \
    time complexity and space complexity. 

    Here is a question:
    {input}"""

Prompt info:

    prompt_infos = [
        {
            "name": "physics", 
            "description": "Good for answering questions about physics", 
            "prompt_template": physics_template
        },
        {
            "name": "math", 
            "description": "Good for answering math questions", 
            "prompt_template": math_template
        },
        {
            "name": "History", 
            "description": "Good for answering history questions", 
            "prompt_template": history_template
        },
        {
            "name": "computer science", 
            "description": "Good for answering computer science questions", 
            "prompt_template": computerscience_template
        }
    ]

Import these 4 from LangChain -
- Multi-prompt chain from router
- LLM router chain from router and llm router
- Router output parer from llm router which parses the LLM output into dictionary tp determine which chain tp use
- Prompt template from prompts

    from langchain.chains.router import MultiPromptChain
    from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
    from langchain.prompts import PromptTemplate

Destination Chains:

Create a dictionary for destination chains that takes each subject as key and LLM chain with model and specific subject prompt as value.

    llm = ChatOpenAI(temperature=0, model=llm_model)


    destination_chains = {}
    for p_info in prompt_infos:
        name = p_info["name"]
        prompt_template = p_info["prompt_template"]
        prompt = ChatPromptTemplate.from_template(template=prompt_template)
        chain = LLMChain(llm=llm, prompt=prompt)
        destination_chains[name] = chain  
        
    destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
    destinations_str = "\n".join(destinations)

Defalt prompt takes only input:

    default_prompt = ChatPromptTemplate.from_template("{input}")
    default_chain = LLMChain(llm=llm, prompt=default_prompt)

Multi prompt template:

    MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
    language model select the model prompt best suited for the input. \
    You will be given the names of the available prompts and a \
    description of what the prompt is best suited for. \
    You may also revise the original input if you think that revising\
    it will ultimately lead to a better response from the language model.

    << FORMATTING >>
    Return a markdown code snippet with a JSON object formatted to look like:
    ```json
    {{{{
        "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
        "next_inputs": string \ a potentially modified version of the original input
    }}}}
    ```

    REMEMBER: The value of “destination” MUST match one of \
    the candidate prompts listed below.\
    If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
    REMEMBER: "next_inputs" can just be the original input \
    if you don't think any modifications are needed.

    << CANDIDATE PROMPTS >>
    {destinations}

    << INPUT >>
    {{input}}

    << OUTPUT (remember to include the ```json)>>"""

    router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
        destinations=destinations_str
    )
    router_prompt = PromptTemplate(
        template=router_template,
        input_variables=["input"],
        output_parser=RouterOutputParser(),
    )

    router_chain = LLMRouterChain.from_llm(llm, router_prompt)

    chain = MultiPromptChain(router_chain=router_chain, 
                            destination_chains=destination_chains, 
                            default_chain=default_chain, verbose=True
                            )